# AutoLyrics Whisper LoRA Fine-Tuning Notebook

Converted from the uploaded Python script.

In [1]:
!pip install torch torchaudio transformers datasets peft bitsandbytes jiwer gradio accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 63.5 MB/s eta 0:00:00


In [2]:
!pip install --upgrade transformers peft torchao accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
import os
import torch
import torchaudio
import gradio as gr
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import load_dataset, Audio
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from jiwer import wer, cer
from dataclasses import dataclass
from typing import Any, Dict, List, Union

## CONFIGURATION & HYPERPARAMETERS

In [4]:
MODEL_NAME = "openai/whisper-small"  #Base model
DATASET_NAME = "gmenon/slt-lyrics-audio"
OUTPUT_DIR = "./whisper-lora-autolyrics"
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
MAX_STEPS = 100

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 1. DATASET LOADING & PREPROCESSING

In [5]:
#Loading the dataset (Hugging Face)
raw_dataset = load_dataset(DATASET_NAME)

#Resampling
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

#Processor (Feature extraction and Text tokenization)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="english", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]

    #Input feautures
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    #Tokenization
    target_text = batch.get("text", batch.get("lyrics", ""))
    batch["labels"] = processor.tokenizer(target_text).input_ids
    return batch

#Select a subset if the dataset is large, or use full splits
train_split = raw_dataset["train"].select(range(min(20, len(raw_dataset["train"]))))
test_split = raw_dataset["test"].select(range(min(10, len(raw_dataset["test"])))) if "test" in raw_dataset else train_split

train_dataset = train_split.map(prepare_dataset, remove_columns=train_split.column_names)
eval_dataset = test_split.map(prepare_dataset, remove_columns=test_split.column_names)

#Data collator to dynamically pad inputs and labels
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/641 [00:00<?, ?B/s]

data/train-00000-of-00012-bdae940d202447(…):   0%|          | 0.00/416M [00:00<?, ?B/s]

data/train-00001-of-00012-959612f33247a9(…):   0%|          | 0.00/445M [00:00<?, ?B/s]

data/train-00002-of-00012-b5ec7053aeb803(…):   0%|          | 0.00/432M [00:00<?, ?B/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Preprocessing dataset (Resampling & Tokenizing)...


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 2. INITIALIZE BASE MODEL & APPLY LoRA

In [6]:
from transformers import BitsAndBytesConfig


#Quantization configuration
quantization_config = None
if device == "cuda":
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

#Model loading using the new quantization_config parameter
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto" if device == "cuda" else None
)

#Applying PEFT(LoRA)
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


--- Initializing Model & LoRA Config ---


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


## 3. TRAINING / FINE-TUNING VIA PEFT

In [7]:
#Base Model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)

#Disable automatic generation config caching during training
model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

#Enabling the input gradients for checkpointing with LoRA
model.enable_input_require_grads()

#Configure LoRA
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)


#Wrapping the base model with PEFT/LoRA wrapper
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


#Evaluation metrics
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_score = 100 * wer(label_str, pred_str)
    return {"wer": wer_score}

#Forcefully strip ANY hidden dataset columns to prevent kwargs TypeErrors
train_dataset = train_dataset.select_columns(["input_features", "labels"])
eval_dataset = eval_dataset.select_columns(["input_features", "labels"])

#Configuring training args
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True if device == "cuda" else False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    label_names=["labels"],
    remove_unused_columns=False,
)

#Initializing the Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

#Training
trainer.train()

#saving the trained LoRA adapter weights
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor successfully saved to {OUTPUT_DIR}")


--- Loading Model and Applying LoRA ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429

--- Defining Evaluation Metric ---

--- Setting Up Trainer ---

--- Starting Training ---


Step,Training Loss,Validation Loss,Wer
50,3.649561,2.077520,2000
100,2.842659,1.650481,2000


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para


--- Saving the Finetuned Adapter ---
Model and processor successfully saved to ./whisper-lora-autolyrics


## 4. EVALUATION MODULE (WER/CER CALCULATION)

In [8]:

def evaluate_predictions(base_model_name, adapter_path, eval_data):
    #Loading both models
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name).to(device)

    peft_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
    peft_model = PeftModel.from_pretrained(peft_model, adapter_path).to(device)

    references = []
    base_preds = []
    lora_preds = []

    #Run comparative inference on test samples
    for sample in eval_data.select(range(min(5, len(eval_data)))):
        input_feats = torch.tensor([sample["input_features"]]).to(device)

        #Ground Truth Text
        ref_ids = [i for i in sample["labels"] if i != -100]
        ref_text = processor.tokenizer.decode(ref_ids, skip_special_tokens=True)
        references.append(ref_text)

        #Zero-Shot Base Inference
        with torch.no_grad():

            base_gen = base_model.generate(input_features=input_feats)
            base_pred = processor.tokenizer.decode(base_gen[0], skip_special_tokens=True)
            base_preds.append(base_pred)

            # LoRA Fine-Tuned Inference
            lora_gen = peft_model.generate(input_features=input_feats)
            lora_pred = processor.tokenizer.decode(lora_gen[0], skip_special_tokens=True)
            lora_preds.append(lora_pred)

    #Compute Error Rates
    base_wer = wer(references, base_preds)
    lora_wer = wer(references, lora_preds)
    base_cer = cer(references, base_preds)
    lora_cer = cer(references, lora_preds)

    print("Results:")
    print(f"Base Model ({base_model_name}):- WER: {base_wer:.4f} %| CER: {base_cer:.4f} %")
    print(f"LoRA Fine-Tuned Model:- WER: {lora_wer:.4f} %| CER: {lora_cer:.4f} %")
    if base_wer > 0:
        improvement = ((base_wer - lora_wer) / base_wer) * 100
        print(f"Relative WER Reduction: {improvement:.2f}%")


evaluate_predictions(MODEL_NAME, OUTPUT_DIR, eval_dataset)


--- Running Performance Evaluation ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


COMPARATIVE PERFORMANCE REPORT
Base Model (openai/whisper-small) -> WER: 28.0000 %| CER: 127.0000 %
LoRA Fine-Tuned Model     -> WER: 5.0000 %| CER: 13.0000 %
Relative WER Reduction: 82.14% (Target: >15%)


## 5. GRADIO DEPLOYMENT DEMO INTERFACE

In [1]:
from transformers import pipeline

inference_model = model.to(device)
inference_model.eval()
inference_model.config.use_cache = True

baseline_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)
baseline_model.eval()
baseline_model.config.use_cache = True

#Chunking
tuned_pipe = pipeline(
    "automatic-speech-recognition",
    model=inference_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    device=device
)

base_pipe = pipeline(
    "automatic-speech-recognition",
    model=baseline_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    device=device
)

def transcribe_long_audio(audio_file_path):
    if audio_file_path is None:
        return "No audio provided.", "No audio provided."


    tuned_result = tuned_pipe(
        audio_file_path,
        generate_kwargs={"max_new_tokens": 225, "num_beams": 3}
    )

    base_result = base_pipe(
        audio_file_path,
        generate_kwargs={"max_new_tokens": 225, "num_beams": 3}
    )

    return tuned_result["text"], base_result["text"]

#Launching Gradio interface
demo = gr.Interface(
    fn=transcribe_long_audio,
    inputs=gr.Audio(type="filepath", label="Upload Full Song"),
    outputs=[
        gr.Textbox(label="AutoLyrics Fine-Tuned Transcription"),
        gr.Textbox(label="Baseline (Un-tuned) Transcription")
    ],
    title="AutoLyrics: Long-Form Singing-Voice ASR"
)

demo.launch(share=True)

KeyboardInterrupt: 